In [1]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
import sys
import os
import hockey.hockey_env as h_env
import numpy as np

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

# Now import your module
from env import HockeyEnv_SB3  # Custom environment based on the provided HockeyEnv
# from hyperparams import TOTAL_TIMESTEPS  # To reuse total timesteps constant
from evaluate import eval_agent

In [2]:
# Create the Hockey environment
def create_env(weak_opponent=False, additional_rewards=None):
    return HockeyEnv_SB3(weak_opponent=False, additional_rewards=None)

In [3]:
# Create a vectorized environment
env = make_vec_env(create_env, n_envs=1)


In [4]:
# Create the PPO model
agent = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    tensorboard_log="./ppo_hockey_tensorboard/"
)

Using cpu device


In [11]:
# Train the model
TOTAL_TIMESTEPS = 500000
agent.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback)

Logging to ./ppo_hockey_tensorboard/PPO_3
Eval num_timesteps=1752, episode_reward=-19.30 +/- 87.93
Episode length: 112.20 +/- 77.07
---------------------------------
| eval/              |          |
|    mean_ep_length  | 112      |
|    mean_reward     | -19.3    |
| time/              |          |
|    total_timesteps | 1752     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 138      |
|    ep_rew_mean     | -57.3    |
| time/              |          |
|    fps             | 2087     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 151         |
|    ep_rew_mean          | -45.2       |
| time/                   |             |
|    fps                  | 1748        |
|    iterations           | 2         

In [12]:
# Save the trained model
agent.save("ppo_hockey_model")

In [5]:
# Load the trained model
agent = PPO.load("ppo_hockey_model")

In [8]:
# Initialize environment and opponents
# eval_env = h_env.HockeyEnv()
# player2 = h_env.BasicOpponent(False)

mean, std = eval_agent(player_left=agent, num_episodes=5)

Episode 1   Reward:   0
Episode 2   Reward:   8
Episode 3   Reward:  -4
Episode 4   Reward:   8
Episode 5   Reward:  -6
Mean Reward: 1.14 ± 6.28
